# 03b — Groundsource: Rainfall Intensity and Area Standardisation

**Purpose:** Derive multi-duration peak rainfall intensities from the IMERG precipitation matrices
and recalculate flood polygon areas using the accurate Mollweide equal-area projection.

**Rainfall intensity method:** For each event, the spatial mean precipitation time series is
computed (averaging only the pixels inside the flood polygon mask). A rolling window of
N half-hour steps is applied, and the maximum of that rolling mean gives the peak intensity
at the corresponding duration. This is computed for 7 durations: 30, 60, 120, 240, 360,
720, and 1440 minutes.

**Area standardisation:** The existing `area_km2` column (from the source dataset) is preserved
as `area_km2_original`. A new `area_km2_mollweide` column is added, derived from the
Mollweide-projected polygon area (`polygon_total_area_m2 / 1e6`).

**Performance:** Intensity calculations are vectorised using NumPy. No row-by-row apply loops
are used. Runtime is measured for each major step.

**Input:**
- `outputs/groundsource_with_imerg.parquet` — events with IMERG matrices (from 03a)

**Output:**
- `outputs/groundsource_rain_master.parquet` — enriched dataset with:
  - `area_km2_original`  : original area from source data (km²)
  - `area_km2_mollweide` : recalculated from equal-area projection (km²)
  - Updated `PFDI_p95`, `PFDI_p99`, `PFDI_max` — recalculated with corrected area
  - `{30,60,120,240,360,720,1440}_max_rainfall_intens` — peak intensity at each duration (mm/hr)

In [ ]:
import pickle
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

In [ ]:
# --- CONFIGURATION ---

INPUT_PATH  = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_with_imerg.parquet"
OUTPUT_PATH = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_rain_master.parquet"

# Duration windows in minutes; each maps to N half-hour IMERG time steps
DURATIONS_MIN = [30, 60, 120, 240, 360, 720, 1440]

# Columns to drop from the final output (intermediate / redundant)
COLS_TO_DROP = ['poly_area_km2']   # replaced by area_km2_mollweide

## 1. Load data

In [ ]:
t_start = time.time()
df = pd.read_parquet(INPUT_PATH)
print(f"Loaded {len(df):,} records in {time.time()-t_start:.1f}s")
print(f"Columns: {list(df.columns)}")

## 2. Area standardisation

In [ ]:
t0 = time.time()

# Preserve the original source area
df = df.rename(columns={'area_km2': 'area_km2_original'})

# Recalculate from the Mollweide-projected polygon area (computed in 01a)
df['area_km2_mollweide'] = df['polygon_total_area_m2'] / 1e6

print(f"Area standardisation done in {time.time()-t0:.2f}s")
print("area_km2_original  stats:")
print(df['area_km2_original'].describe())
print("\narea_km2_mollweide stats:")
print(df['area_km2_mollweide'].describe())

## 3. Recalculate PFDI with corrected area

In [ ]:
t0 = time.time()

area = df['area_km2_mollweide'].values
valid = area > 0

for pfdi_col, upa_col in [('PFDI_p95', 'upa_p95'),
                           ('PFDI_p99', 'upa_p99'),
                           ('PFDI_max', 'upa_max')]:
    df[pfdi_col] = np.where(valid, df[upa_col].values / area, np.nan)

print(f"PFDI recalculation done in {time.time()-t0:.2f}s")
for col in ['PFDI_p95', 'PFDI_p99', 'PFDI_max']:
    pct = (df[col] <= 1.0).mean() * 100
    print(f"  {col} ≤ 1 (pluvial): {pct:.1f}%")

## 4. Compute multi-duration peak rainfall intensities

For each event:
1. Average the IMERG pixels inside the flood mask → spatial mean time series (1-D, one value per 30-min step)
2. Apply a rolling window mean of width = duration / 30 min
3. Record the maximum of the rolling mean as the peak intensity for that duration

In [ ]:
def compute_spatial_mean_series(matrix_bytes, mask_bytes):
    """
    Compute the spatial mean precipitation time series for one event.

    Args:
        matrix_bytes : bytes — pickled 3-D numpy array (T × H × W) of rainfall [mm/hr]
        mask_bytes   : bytes — pickled 2-D binary array (H × W) — 1 inside polygon

    Returns:
        1-D numpy array of length T, or None if the mask is empty.
    """
    matrix = pickle.loads(matrix_bytes)   # shape: (T, H, W)
    mask   = pickle.loads(mask_bytes)     # shape: (H, W)

    n_pixels = mask.sum()
    if n_pixels == 0:
        return None

    # Broadcast mask across time dimension and compute mean over spatial pixels
    masked  = matrix * mask[np.newaxis, :, :]   # (T, H, W), zeros outside polygon
    series  = masked.sum(axis=(1, 2)) / n_pixels  # (T,)
    return series


def peak_rolling_mean(series, n_steps):
    """
    Return the maximum of a rolling window mean over `n_steps` half-hour steps.

    A window of 1 step = 30-min peak intensity.
    A window of 2 steps = maximum 60-min mean intensity. Etc.
    """
    if series is None or len(series) < n_steps:
        return np.nan
    # Compute rolling mean using a convolution filter (fast, no loops)
    kernel      = np.ones(n_steps) / n_steps
    rolling_avg = np.convolve(series, kernel, mode='valid')
    return float(rolling_avg.max()) if len(rolling_avg) > 0 else np.nan

In [ ]:
t0 = time.time()
n  = len(df)

# Pre-compute spatial mean series for all events (one pass over the data)
print("Step 1/2: Computing spatial mean precipitation series for all events...")
series_list = [
    compute_spatial_mean_series(m, mk)
    for m, mk in zip(df['imerg_matrix'], df['imerg_mask'])
]
print(f"  Done in {time.time()-t0:.1f}s")

# For each duration, apply the rolling window and find the peak
print("Step 2/2: Computing peak intensities for each duration...")
t1 = time.time()
for dur_min in DURATIONS_MIN:
    n_steps  = dur_min // 30   # number of 30-min IMERG steps in this window
    col_name = f"{dur_min}_max_rainfall_intens"
    df[col_name] = [
        peak_rolling_mean(s, n_steps) for s in series_list
    ]
    print(f"  {dur_min:>5} min  (window={n_steps} steps)  — "
          f"non-null: {df[col_name].notna().sum():,}  "
          f"median: {df[col_name].median():.4f} mm/hr")

print(f"Intensity computation done in {time.time()-t1:.1f}s")

## 5. Clean up and save

In [ ]:
# Drop intermediate columns that are superseded by the new ones
cols_present = [c for c in COLS_TO_DROP if c in df.columns]
df = df.drop(columns=cols_present)
print(f"Dropped columns: {cols_present}")
print(f"Final columns ({len(df.columns)}): {list(df.columns)}")

df.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved {len(df):,} rows → {OUTPUT_PATH}")
print(f"Total runtime: {time.time()-t_start:.1f}s")

## 6. Quick sanity check on intensity columns

In [ ]:
import matplotlib.pyplot as plt

intensity_cols = [f"{d}_max_rainfall_intens" for d in DURATIONS_MIN]
medians = [df[c].median() for c in intensity_cols]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(DURATIONS_MIN, medians, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Median peak intensity (mm/hr)')
ax.set_title('Intensity–Duration Curve (median over all events) — Groundsource')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.set_xticks(DURATIONS_MIN)
ax.set_xticklabels([str(d) for d in DURATIONS_MIN])
plt.tight_layout()
plt.show()